# Análise dos probes do world model — seeds 0 e 2

Este notebook consolida os resultados dos probes sobre as camadas `h1`–`h4`, usa a **seed 2** (run W&B `vednsvgo`) como foco principal e compara sua representação com a seed 0. Também verifica integridade, compara Ridge e MLP, analisa incerteza e contrasta a rodada corrigida com a legada.

**Leitura das métricas:** R² próximo de 1 indica alta previsibilidade; R² próximo de 0 equivale aproximadamente a prever a média; R² negativo indica generalização pior que a média. O Ridge mede acessibilidade linear. O ganho `MLP − Ridge` indica informação não linear adicional, desde que seja estável entre seeds.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 120)

def find_output_dir():
    candidates = [Path('papers/probe_outputs'), Path('probe_outputs'), Path.cwd() / 'papers' / 'probe_outputs']
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Não encontrei papers/probe_outputs. Execute o notebook a partir do repositório.')

OUTPUT_DIR = find_output_dir()
RUN_TAG = 'scaled_mlp_v2'
FOCUS_SEED = 2
COMPARISON_SEEDS = [0, 2]

def one_file(pattern):
    matches = sorted(OUTPUT_DIR.glob(pattern))
    if len(matches) != 1:
        raise RuntimeError(f'Esperava 1 arquivo para {pattern}, encontrei {len(matches)}: {matches}')
    return matches[0]

paths = {
    'probe': one_file(f'*__seed{FOCUS_SEED}__*__{RUN_TAG}__probe_results.csv'),
    'diagnostic': one_file(f'*__seed{FOCUS_SEED}__*__{RUN_TAG}__diagnostic_probe_results.csv'),
    'controls': one_file(f'*__seed{FOCUS_SEED}__*__{RUN_TAG}__controls.csv'),
    'summary': one_file(f'*__seed{FOCUS_SEED}__*__{RUN_TAG}__summary.json'),
}
probe = pd.read_csv(paths['probe'])
diagnostic = pd.read_csv(paths['diagnostic'])
controls = pd.read_csv(paths['controls'])
summary = json.loads(paths['summary'].read_text())

print('Diretório:', OUTPUT_DIR)
print('Seed em foco:', FOCUS_SEED)
for name, path in paths.items():
    print(f'{name:>10}: {path.name}')

## 1. Integridade da execução e tempo

Além de conferir os quatro artefatos, verificamos quantas combinações camada/alvo foram avaliadas, quais foram descartadas por serem constantes e se há métricas ausentes inesperadas.

In [ ]:
elapsed = float(summary['elapsed_seconds'])
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60

integrity = pd.DataFrame({
    'item': ['transições totais', 'transições usadas', 'seeds do MLP', 'linhas principais',
             'probes principais válidos', 'targets constantes descartados', 'diagnósticos válidos',
             'tempo total (s)', 'tempo total formatado'],
    'valor': [summary['num_transitions_total'], summary['num_transitions_used'],
              str(summary['probe_seeds']), len(probe), (probe.status == 'ok').sum(),
              (probe.status == 'skipped_constant_target').sum(),
              (diagnostic.status == 'ok').sum(), round(elapsed, 2),
              f'{hours:02d}:{minutes:02d}:{seconds:05.2f}']
})
display(integrity)

assert len(probe) == 80, 'Número inesperado de linhas no resultado principal.'
assert len(diagnostic) == 20, 'Número inesperado de linhas no diagnóstico.'
assert (diagnostic.status == 'ok').all(), 'Há diagnóstico incompleto.'
assert probe.loc[probe.status == 'ok', ['ridge_r2', 'mlp_r2']].notna().all().all()
print('\n✓ Artefatos retornados e resultados válidos estão completos.')

In [ ]:
constant_targets = (probe.loc[probe.status == 'skipped_constant_target',
                              ['target', 'target_dim_original', 'target_dim_constant']]
                    .drop_duplicates().reset_index(drop=True))
display(Markdown('### Targets descartados corretamente por variância zero'))
display(constant_targets)
print('Esses targets explicam os R² exatamente 1,0 da rodada antiga; não eram codificação perfeita.')

## 2. Representações aprendidas ao longo das camadas

Os alvos abaixo resumem estado atual, próxima observação, mudança de estado, deslocamento, reward, comando e memória de ações.

In [ ]:
valid = probe.query("status == 'ok'").copy()
core_targets = ['obs_sanity', 'next_obs', 'delta_obs', 'delta_xy', 'reward',
                'info_command', 'info_last_act', 'info_last_last_act']
core = valid[valid.target.isin(core_targets)].copy()

fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
for ax, target in zip(axes.flat, core_targets):
    data = core[core.target == target].set_index('layer').reindex(['h1', 'h2', 'h3', 'h4'])
    ax.plot(data.index, data.ridge_r2, marker='o', label='Ridge')
    ax.errorbar(data.index, data.mlp_r2, yerr=data.mlp_r2_std,
                marker='s', capsize=3, label='MLP')
    ax.axhline(0, color='black', linewidth=.8)
    ax.set_title(target)
    ax.set_ylabel('R²')
axes[0, 0].legend()
fig.suptitle('Acessibilidade linear e não linear por camada', fontsize=15)
fig.tight_layout()
plt.show()

In [ ]:
best_rows = []
for target in core_targets:
    data = core[core.target == target]
    ridge_best = data.loc[data.ridge_r2.idxmax()]
    mlp_best = data.loc[data.mlp_r2.idxmax()]
    best_rows.append({
        'target': target,
        'melhor camada Ridge': ridge_best.layer,
        'melhor Ridge R²': ridge_best.ridge_r2,
        'melhor camada MLP': mlp_best.layer,
        'melhor MLP R²': mlp_best.mlp_r2,
        'MLP std': mlp_best.mlp_r2_std,
    })
best = pd.DataFrame(best_rows)
display(best.style.format({'melhor Ridge R²': '{:.3f}', 'melhor MLP R²': '{:.3f}', 'MLP std': '{:.3f}'}))

**Interpretação:** as primeiras camadas preservam melhor estado, comando e ações recentes. Com a profundidade, o Ridge recupera melhor `delta_obs`, `delta_xy` e reward, enquanto a observação atual e o comando ficam menos linearmente explícitos. Isso é consistente com uma transformação de uma representação próxima da entrada para outra mais alinhada à saída preditiva do world model.

O MLP corrige os valores patológicos da rodada antiga. Seus desvios entre seeds são geralmente pequenos. Ele revela não linearidade relevante em `h4` para posição atual, próxima posição e comando, mas não supera o Ridge em `delta_obs` ou `delta_xy`; portanto, a dinâmica incremental aprendida parece estar mais linearmente acessível do que escondida numa codificação não linear.

## 3. Quanto a camada interna acrescenta à entrada bruta?

In [ ]:
delta = valid[valid.target == 'delta_obs'][['layer', 'ridge_r2', 'mlp_r2', 'mlp_r2_std']].copy()
input_control = float(controls.loc[controls.control == 'input_obs_action', 'ridge_r2'].iloc[0])
delta['ganho Ridge sobre entrada'] = delta.ridge_r2 - input_control
display(delta.style.format({c: '{:.3f}' for c in delta.columns if c != 'layer'}))

display(controls[['control', 'ridge_r2', 'ridge_nrmse']].style.format({'ridge_r2': '{:.3f}', 'ridge_nrmse': '{:.3f}'}))

**Interpretação:** a entrada bruta obtém aproximadamente R² 0,62 para `delta_obs`, enquanto `h4` chega a aproximadamente 0,71. Há informação preditiva adicional, mas o ganho é moderado. Labels embaralhados e features aleatórias permanecem perto ou abaixo de zero, apoiando que o sinal não é artefato da dimensionalidade.

## 4. Diagnóstico: incerteza interna versus erro realizado

In [ ]:
diag_targets = ['wm_obs_error_norm', 'wm_reward_abs_error', 'unc_epi_mean', 'unc_ale_max', 'unc_total_var']
fig, axes = plt.subplots(1, len(diag_targets), figsize=(20, 4), sharex=True)
for ax, target in zip(axes, diag_targets):
    data = diagnostic[diagnostic.target == target].set_index('layer').reindex(['h1', 'h2', 'h3', 'h4'])
    ax.plot(data.index, data.ridge_r2, marker='o', label='Ridge')
    ax.errorbar(data.index, data.mlp_r2, yerr=data.mlp_r2_std,
                marker='s', capsize=3, label='MLP')
    ax.axhline(0, color='black', linewidth=.8)
    ax.set_title(target)
    ax.set_ylabel('R²')
axes[0].legend()
fig.suptitle('Diagnósticos do world model', fontsize=15)
fig.tight_layout()
plt.show()

diag_summary = diagnostic.pivot(index='target', columns='layer', values=['ridge_r2', 'mlp_r2'])
display(diag_summary.round(3))

**Interpretação principal:** a incerteza produzida pelo próprio ensemble é altamente decodificável, chegando a R² ≈ 0,96 para variância total em `h4`. Em contraste, o erro efetivamente realizado continua fracamente previsível: no melhor caso, R² ≈ 0,23 para erro da observação e ≈ 0,13 para erro do reward com Ridge. O MLP não fecha essa lacuna.

Portanto, o hidden state representa muito bem a estrutura interna de incerteza do ensemble, mas isso não demonstra calibração contra o erro real. Como hidden state e incerteza são produzidos pelo mesmo modelo, a alta previsibilidade da incerteza pode ser parcialmente estrutural.

## 5. Onde o MLP acrescenta informação não linear?

In [ ]:
nonlinear = valid[['layer', 'target', 'ridge_r2', 'mlp_r2', 'mlp_r2_std', 'delta_mlp_minus_ridge']].copy()
top_gain = nonlinear.sort_values('delta_mlp_minus_ridge', ascending=False).head(12)
top_loss = nonlinear.sort_values('delta_mlp_minus_ridge').head(12)

display(Markdown('### Maiores ganhos do MLP'))
display(top_gain.style.format({c: '{:.3f}' for c in top_gain.columns if c not in ['layer', 'target']}))
display(Markdown('### Casos em que o Ridge foi melhor'))
display(top_loss.style.format({c: '{:.3f}' for c in top_loss.columns if c not in ['layer', 'target']}))

Os maiores ganhos não lineares aparecem principalmente para comando e posição nas camadas profundas, além das incertezas em `h1`/`h2`. Já mudança de estado e deslocamento favorecem o Ridge. R² negativo para duração/magnitude de perturbações e eventos futuros significa ausência de evidência de decodificação, não evidência de que a rede tenha aprendido valores negativos.

## 6. Antes e depois da correção do MLP

In [ ]:
legacy_candidates = [p for p in OUTPUT_DIR.glob('*__probe_results.csv') if RUN_TAG not in p.name]
if legacy_candidates:
    legacy = pd.read_csv(sorted(legacy_candidates)[0])
    comparison_targets = ['reward', 'delta_obs', 'next_obs', 'info_command']
    before = legacy[legacy.target.isin(comparison_targets)][['layer', 'target', 'mlp_r2']].rename(columns={'mlp_r2': 'MLP legado'})
    after = valid[valid.target.isin(comparison_targets)][['layer', 'target', 'mlp_r2', 'mlp_r2_std']].rename(columns={'mlp_r2': 'MLP corrigido'})
    comparison = before.merge(after, on=['layer', 'target'])
    display(comparison.style.format({'MLP legado': '{:.3f}', 'MLP corrigido': '{:.3f}', 'mlp_r2_std': '{:.3f}'}))
else:
    print('Rodada legada não encontrada; comparação omitida.')

A correção elimina os R² extremamente negativos do MLP em targets de pequena escala. O exemplo mais claro é reward: a rodada antiga chegava a valores negativos em várias camadas, enquanto a corrigida fica aproximadamente entre 0,85 e 0,90. Isso confirma que o comportamento anterior era majoritariamente um problema de treinamento/escala do probe.

## 7. Custo computacional

In [ ]:
timing = pd.concat([
    valid.assign(grupo='principal'),
    diagnostic.assign(grupo='diagnóstico')
], ignore_index=True)

timing_summary = (timing.groupby('grupo')
                  .agg(probes=('target', 'size'),
                       tempo_probes_s=('probe_elapsed_seconds', 'sum'),
                       media_por_probe_s=('probe_elapsed_seconds', 'mean'),
                       máximo_por_probe_s=('probe_elapsed_seconds', 'max'),
                       épocas_MLP_médias=('mlp_epochs', 'mean'))
                  .reset_index())
display(timing_summary.style.format({c: '{:.2f}' for c in timing_summary.columns if c not in ['grupo', 'probes']}))

slowest = timing.nlargest(10, 'probe_elapsed_seconds')[['grupo', 'layer', 'target', 'probe_elapsed_seconds', 'mlp_epochs']]
display(Markdown('### Dez probes mais demorados'))
display(slowest.style.format({'probe_elapsed_seconds': '{:.2f}', 'mlp_epochs': '{:.1f}'}))

## 8. Comparação entre seeds 0 e 2

A seed 2 corresponde ao checkpoint da run W&B `vednsvgo`. A comparação abaixo usa exatamente os mesmos dados amostrados, split e seeds dos probes, isolando principalmente a variação causada pelo treinamento do world model.

In [ ]:
runs = {}
for seed in COMPARISON_SEEDS:
    seed_paths = {
        'probe': one_file(f'*__seed{seed}__*__{RUN_TAG}__probe_results.csv'),
        'diagnostic': one_file(f'*__seed{seed}__*__{RUN_TAG}__diagnostic_probe_results.csv'),
        'controls': one_file(f'*__seed{seed}__*__{RUN_TAG}__controls.csv'),
        'summary': one_file(f'*__seed{seed}__*__{RUN_TAG}__summary.json'),
    }
    runs[seed] = {
        'probe': pd.read_csv(seed_paths['probe']),
        'diagnostic': pd.read_csv(seed_paths['diagnostic']),
        'controls': pd.read_csv(seed_paths['controls']),
        'summary': json.loads(seed_paths['summary'].read_text()),
    }

def compare_seeds(kind):
    frames = []
    for seed, run in runs.items():
        frame = run[kind].copy()
        frame['world_model_seed'] = seed
        if 'status' in frame:
            frame = frame[frame.status == 'ok']
        frames.append(frame)
    both = pd.concat(frames, ignore_index=True)
    left = both[both.world_model_seed == 0]
    right = both[both.world_model_seed == 2]
    keys = ['layer', 'target'] if kind != 'controls' else ['control', 'target']
    out = left.merge(right, on=keys, suffixes=('_seed0', '_seed2'))
    out['ridge_diff_seed2_minus_seed0'] = out.ridge_r2_seed2 - out.ridge_r2_seed0
    if kind != 'controls':
        out['mlp_diff_seed2_minus_seed0'] = out.mlp_r2_seed2 - out.mlp_r2_seed0
    return out

probe_comparison = compare_seeds('probe')
diagnostic_comparison = compare_seeds('diagnostic')
control_comparison = compare_seeds('controls')

comparison_summary = pd.DataFrame([
    {
        'grupo': 'probes principais',
        'diferença média Ridge': probe_comparison.ridge_diff_seed2_minus_seed0.mean(),
        'diferença absoluta média Ridge': probe_comparison.ridge_diff_seed2_minus_seed0.abs().mean(),
        'diferença média MLP': probe_comparison.mlp_diff_seed2_minus_seed0.mean(),
        'diferença absoluta média MLP': probe_comparison.mlp_diff_seed2_minus_seed0.abs().mean(),
    },
    {
        'grupo': 'diagnósticos',
        'diferença média Ridge': diagnostic_comparison.ridge_diff_seed2_minus_seed0.mean(),
        'diferença absoluta média Ridge': diagnostic_comparison.ridge_diff_seed2_minus_seed0.abs().mean(),
        'diferença média MLP': diagnostic_comparison.mlp_diff_seed2_minus_seed0.mean(),
        'diferença absoluta média MLP': diagnostic_comparison.mlp_diff_seed2_minus_seed0.abs().mean(),
    },
])
display(comparison_summary.style.format({c: '{:+.4f}' for c in comparison_summary.columns if c != 'grupo'}))

In [ ]:
comparison_targets = ['delta_obs', 'next_obs', 'reward', 'obs_sanity',
                      'delta_xy', 'info_command', 'info_last_act']
selected = probe_comparison[probe_comparison.target.isin(comparison_targets)].copy()

fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
for ax, target in zip(axes.flat, comparison_targets):
    data = selected[selected.target == target].set_index('layer').reindex(['h1', 'h2', 'h3', 'h4'])
    ax.plot(data.index, data.ridge_r2_seed0, marker='o', label='Seed 0')
    ax.plot(data.index, data.ridge_r2_seed2, marker='s', linestyle='--', label='Seed 2')
    ax.axhline(0, color='black', linewidth=.8)
    ax.set_title(target)
    ax.set_ylabel('Ridge R²')
axes[0, 0].legend()
axes.flat[-1].axis('off')
fig.suptitle('Comparação das representações: seed 0 × seed 2', fontsize=15)
fig.tight_layout()
plt.show()

display(selected[['layer', 'target', 'ridge_r2_seed0', 'ridge_r2_seed2',
                  'ridge_diff_seed2_minus_seed0', 'mlp_r2_seed0', 'mlp_r2_seed2',
                  'mlp_diff_seed2_minus_seed0']]
        .style.format({c: '{:+.3f}' for c in selected.columns if c not in ['layer', 'target']}))

In [ ]:
diag_selected = diagnostic_comparison[['layer', 'target', 'ridge_r2_seed0', 'ridge_r2_seed2',
                                           'ridge_diff_seed2_minus_seed0', 'mlp_r2_seed0',
                                           'mlp_r2_seed2', 'mlp_diff_seed2_minus_seed0']]
display(diag_selected.style.format({c: '{:+.3f}' for c in diag_selected.columns if c not in ['layer', 'target']}))

run_times = pd.DataFrame([
    {'seed': seed, 'tempo_s': run['summary']['elapsed_seconds'],
     'tempo_min': run['summary']['elapsed_seconds'] / 60}
    for seed, run in runs.items()
])
display(run_times.style.format({'tempo_s': '{:.2f}', 'tempo_min': '{:.2f}'}))

**Interpretação entre seeds:** as representações são extremamente semelhantes. Nos probes principais, a diferença absoluta média é de aproximadamente 0,009 em Ridge e 0,007 em MLP; as diferenças médias assinadas ficam próximas de zero. A seed 2 reproduz a mesma progressão `h1 → h4`, chega a Ridge R² ≈ 0,715 em `delta_obs` e ≈ 0,928 em reward. Isso reduz bastante a probabilidade de os resultados da seed 0 serem acaso ou uma inicialização excepcional.

A seed 2 melhora pontualmente `delta_xy` em `h2` e reward, mas não existe superioridade sistemática de uma seed. Os diagnósticos também repetem a conclusão: incerteza interna é altamente decodificável, enquanto erro realizado permanece pouco previsível.

## 9. Conclusões

1. **As duas execuções retornaram completas:** cada seed usou 10.000 transições, três seeds do MLP, 68 probes principais válidos e 20 diagnósticos. A seed 0 levou 18 min 27 s e a seed 2 levou 18 min 21 s.
2. **Os falsos R² perfeitos foram resolvidos:** `pert_dir`, `pert_steps` e `steps_since_last_pert` são constantes nesta amostra e foram corretamente descartados.
3. **Existe uma progressão representacional:** `h1` preserva melhor estado/comando/histórico; `h4` lineariza melhor mudança de estado, deslocamento e reward.
4. **O MLP agora é utilizável:** normalizar o target e validar por episódio removeu as falhas extremas. Os desvios pequenos entre seeds indicam resultados estáveis nesta amostra.
5. **A não linearidade é seletiva:** o MLP acrescenta bastante para comando e posição em `h4`, mas Ridge permanece melhor para `delta_obs` e `delta_xy`.
6. **Incerteza interna não equivale a erro real:** incertezas do ensemble são quase perfeitamente recuperáveis, enquanto os erros realizados continuam pouco previsíveis.
7. **A run `vednsvgo` foi validada pelo probe correto:** a seed 2 reproduz quase exatamente os padrões da seed 0, apoiando que o aprendizado é estável entre inicializações.
8. **Limitação:** ainda há apenas um dataset, um membro do ensemble por world model e uma amostra de transições. Para afirmações gerais, repetir em mais membros, model seeds e datasets.